## Pathway Over-Representation Analysis

- Pathway over-representation analysis (ORA) was performed on selected protein sets identified from differential analyses.
- Custom pathway databases were downloaded (see below) for use in enrichment testing.
- Enrichment analyses were conducted to identify biological pathways significantly over-represented relative to background gene sets.
- Statistical significance was assessed using fisher's exact test with multiple testing correction applied.
- Results are summarized using ranked pathway tables and enrichment visualizations.

In [4]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
  library(ggplot2)
  library(purrr)
  library(stringr)
  library(tibble) 
  library(WebGestaltR)
})

In [5]:
## Conduct pathway analyses using
## the webgestaltR package
## and the following 4 databases 

## MSIGDG
# download.file("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=MSigDB_Hallmark_2020",
#               "../../data/gmt/MSigDB_Hallmark_2020.gmt", mode = "wb")

## Reactome
# download.file("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=Reactome_Pathways_2024",
#               "../../data/gmt/Reactome_Pathways_2024.gmt", mode = "wb")

## Encode
# download.file("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X",
#               "../../data/gmt/ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X.gmt", mode = "wb")

## KEGG
# download.file("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=KEGG_2021_Human",
#               "../../data/gmt/KEGG_2021_Human.gmt", mode = "wb")

simplifiedORA <- function(database, 
                          contrast='PreTx-EI', 
                          cutoff){

	contrast_matrix = pvalMat[pvalMat$Contrast==contrast,]
    deps = contrast_matrix[contrast_matrix$adjP < cutoff,]$Assay

    if(length(deps)>0){
        
		res = WebGestaltR::WebGestaltR(enrichMethod = 'ORA', 
                                 organism ='hsapiens', 
                                 enrichDatabase = database,
                                 interestGene = deps, 
                                 interestGeneType ="genesymbol",
                                 referenceSet = 'genome_protein-coding',
                                 fdrThr = 0.05,
                                 minNum = 10,
                                 maxNum = 500,
                                 isOutput=FALSE)
        
        res$Contrast = contrast
        res
    } else{
        print('no DEPs')
        }
}


In [6]:
## Load DEPs 
longitudinal_deps <- fread("../../data/olink/output/plasma_longitudinal_pairwise_deps.csv", data.table = F)
healthy_comparison_deps <- fread("../../data/olink/output/plasma_longitudinal_comparisons_to_healthy.csv", data.table=F)

In [7]:
simplifiedORA_external_gmt <- function(database_path, 
                                       pvalMat, 
                                       contrast, 
                                       cutoff=0.05){

	contrast_matrix = pvalMat[pvalMat$Contrast==contrast,]
    deps = contrast_matrix[contrast_matrix$adjP < cutoff,]$Assay
    ref_genes = contrast_matrix$Assay

    if(length(deps)>0){
        
            ora_results <- WebGestaltR(
              enrichMethod = "ORA",   # Specify the enrichment method
              interestGene = deps,   # Your list of genes
              interestGeneType="genesymbol",
              enrichDatabase="others",
              enrichDatabaseFile=database_path,
              enrichDatabaseType='genesymbol',
              referenceSet = 'genome_protein-coding',
              refGene = ref_genes,
              referenceGeneType='genesymbol',
              fdrMethod = "BH",       
                                 fdrThr = 0.05,
                                 minNum = 10,
                                 maxNum = 500,
                isOutput=FALSE
            )

            
        ora_results$Contrast = contrast
        ora_results
    } else{
        print('no DEPs')
        }
    }

gmt_MSigDB = '../../data/gmt/MSigDB_Hallmark_2020.gmt'
gmt_KEGG = '../../data/gmt/KEGG_2021_Human.gmt'
gmt_Encode = '../../data/gmt/ENCODE_and_ChEA_Consensus_TFs_from_ChIP-X.gmt'
gmt_Reactome = '../../data/gmt/Reactome_Pathways_2024.gmt'

# Run ORA on Longitudinal Differential Proteins

In [8]:
sink('tmpfile')
contrasts= c('PreTx-PI2C','PreTx-EI','EI-ASCT60d','EI-ASCT1y', 'ASCT60d-ASCT1y') 
## Kegg Pathways
keggPathways = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_KEGG, pvalMat = longitudinal_deps, contrast=x)), fill=T)
keggPathways$Database='KEGG'

## Reactome Pathways
ReactomePathways = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_Reactome, pvalMat = longitudinal_deps, contrast=x)), fill=T)
ReactomePathways$Database='Reactome'                             

## MSigDB Pathways
msigdbPathways = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_MSigDB, pvalMat = longitudinal_deps,contrast=x)),fill=T)
msigdbPathways$Database='MSigDB'
                           
## Encode Pathways
encodePathways = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_Encode,pvalMat = longitudinal_deps, contrast=x)), fill=T)
encodePathways$Database='Encode'                           

sink()   

Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”


In [9]:
### Set factors to order results by clinical comparisons and combine all results
contrast_levels = c('PreTx-PI2C','PreTx-EI','EI-ASCT60d','EI-ASCT1y','ASCT60d-ASCT1y')
keggPathways$Contrast <- factor(keggPathways$Contrast, levels=contrast_levels)
ReactomePathways$Contrast <- factor(ReactomePathways$Contrast, levels=contrast_levels)
msigdbPathways$Contrast <- factor(msigdbPathways$Contrast, levels=contrast_levels)
encodePathways$Contrast <- factor(encodePathways$Contrast, levels=contrast_levels)

combined_pathways <- rbind(keggPathways,
          ReactomePathways,
          msigdbPathways,
          encodePathways, fill=T)

head(combined_pathways,2)

geneSet,link,size,overlap,expect,enrichmentRatio,pValue,FDR,overlapId,userId,Contrast,Database
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<fct>,<chr>
Cytokine-cytokine receptor interaction,,295,11,1.947321,5.648787,2.540366e-06,7.875134e-04,6367;6375;8741;2658;3588;4982;23495;84957;163702;4049;10673,RELT;GDF2;TNFSF13B;IFNLR1;LTA;TNFRSF13B;TNFSF13;TNFRSF11B;CCL22;IL10RB;XCL1,PreTx-PI2C,KEGG
Cytokine-cytokine receptor interaction,,295,15,2.100052,7.142681,1.124313e-09,3.485371e-07,3953;6367;6373;6375;6363;3570;8741;3588;4982;115650;23495;3568;163702;654;10673,CCL19;CXCL11;BMP6;LEPR;TNFSF13B;IL6R;TNFRSF13C;IFNLR1;IL5RA;TNFRSF13B;TNFSF13;TNFRSF11B;CCL22;IL10RB;XCL1,PreTx-EI,KEGG


# Run ORA-Pathway Enrichment on DEPs from Healthy Comparisons

In [10]:
sink('tmpfiile')
## Kegg Pathways
contrasts= c('PreTx-Healthy','PI2C-Healthy','EI-Healthy','ASCT60d-Healthy', 'ASCT1y-Healthy','ASCT2y-Healthy') 
keggPathways_healthy = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_KEGG, pvalMat = healthy_comparison_deps, contrast=x)), fill=T)
keggPathways_healthy$Database='Kegg'                                 

## Reactome Pathways
ReactomePathways_healthy = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_Reactome, pvalMat = healthy_comparison_deps, contrast=x)),fill=T)
ReactomePathways_healthy$Database='Reactome'                                     

## MSigDB Pathways
msigdbPathways_healthy = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_MSigDB, pvalMat = healthy_comparison_deps, contrast=x)),fill=T)
msigdbPathways_healthy$Database='MSigDB'
                                   
## Encode Pathways
encodePathways_healthy = rbindlist(lapply(contrasts,
       function(x) simplifiedORA_external_gmt(database=gmt_Encode,pvalMat = healthy_comparison_deps, contrast=x)),fill=T)
encodePathways_healthy$Database='Encode'                                   

keggPathways_healthy$Contrast <- factor(keggPathways_healthy$Contrast, levels=contrasts)
ReactomePathways_healthy$Contrast <- factor(ReactomePathways_healthy$Contrast, levels=contrasts)
msigdbPathways_healthy$Contrast <- factor(msigdbPathways_healthy$Contrast, levels=contrasts)
encodePathways_healthy$Contrast <- factor(encodePathways_healthy$Contrast, levels=contrasts)
sink()   

Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”
Warning message in oraEnrichment(interestGeneList, referenceGeneList, geneSet, minNum = minNum, :
“No significant gene set is identified based on FDR 0.05!”


## Save results to file

In [12]:
combined_pathways_healthy = rbind(keggPathways_healthy,
                                  encodePathways_healthy,
                                  msigdbPathways_healthy,
                                  ReactomePathways_healthy)

In [13]:
write.csv(combined_pathways,
          file='../../data/olink/output/longitudinal_enriched_pathways.csv')

In [16]:
write.csv(combined_pathways_healthy,
          file='../../data/olink/output/cross-sectional-pathways-on-healthy-comparisons.csv')
write.csv(msigdbPathways_healthy,
          file='../../data/olink/output/cross-sectional-msigdb-pathways-on-healthy-comparisons.csv')